1. Setup and Video Initialization

In [62]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.linear_model import RANSACRegressor

# Define paths to your specific project videos 
video_dir = 'data/videos/'
video_files = ['project_video.mp4', 'challenge_video.mp4', 'harder_challenge_video.mp4']
video_file = video_files[2]
current_video = os.path.join(video_dir, video_file)

cap = cv2.VideoCapture(current_video)
print(f"SUCCESS: Loaded {video_file} at {int(cap.get(cv2.CAP_PROP_FPS))} FPS")

SUCCESS: Loaded harder_challenge_video.mp4 at 25 FPS


2. Pre-processing Functions

These functions isolate lane line pixels by filtering for specific colors (white/yellow) and defining a Region of Interest (ROI) to exclude irrelevant parts of the image like trees or the sky.

In [ ]:
def preprocess_frame(frame):
    """Isolates white/yellow lane pixels using HLS masking and applies Gaussian blur."""
    # 1. Convert to HLS color space
    hls = cv2.cvtColor(frame, cv2.COLOR_RGB2HLS)
    
    # 2. Define White Lane Mask (High Lightness)
    lower_white = np.array([0, 200, 0])
    upper_white = np.array([180, 255, 255])
    white_mask = cv2.inRange(hls, lower_white, upper_white)
    
    # 3. Define Yellow Lane Mask (Specific Hue + High Saturation)
    lower_yellow = np.array([15, 0, 100])
    upper_yellow = np.array([35, 255, 255])
    yellow_mask = cv2.inRange(hls, lower_yellow, upper_yellow)
    
    # 4. Combine masks and apply to original image or grayscale
    combined_mask = cv2.bitwise_or(white_mask, yellow_mask)
    
    # Apply mask to grayscale to maintain edge detection compatibility
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    masked_gray = cv2.bitwise_and(gray, combined_mask)
    
    # 5. Apply Gaussian smoothing
    kernel_size = 5
    blur_gray = cv2.GaussianBlur(masked_gray, (kernel_size, kernel_size), 0)
    
    return blur_gray

def get_roi_mask(image):
    """Applies a trapezoidal mask to focus detection on the road area."""
    # Define a road region-of-interest (ROI) 
    mask = np.zeros_like(image)
    imshape = image.shape
    
    # Vertices for a trapezoidal ROI covering the highway lane 
    vertices = np.array([[(0, imshape[0]), (450, 320), (500, 320), (imshape[1], imshape[0])]], dtype=np.int32)
    
    cv2.fillPoly(mask, vertices, 255)
    masked_image = cv2.bitwise_and(image, mask)
    return masked_image

3. Geometric Transformations

To accurately measure lane curvature, the image is transformed from a dashboard perspective to a "bird's-eye view" using perspective mapping.

In [ ]:
def get_perspective_transform(image):
    """Warps an image to a top-down bird's-eye view using predefined source/destination points."""
    img_size = (image.shape[1], image.shape[0])
    
    # Define source points (trapezoid from original image)
    src = np.float32([[200, 720], [1100, 720], [595, 450], [685, 450]])
    
    # Define destination points (rectangle for bird's-eye view)
    dst = np.float32([[300, 720], [980, 720], [300, 0], [980, 0]])
    
    # Compute the transformation matrices
    M = cv2.getPerspectiveTransform(src, dst)
    Minv = cv2.getPerspectiveTransform(dst, src)
    
    warped = cv2.warpPerspective(image, M, img_size, flags=cv2.INTER_LINEAR)
    return warped, Minv
def get_warp_matrices(img_shape):
    """Calculates transformation matrices (M and Minv) for perspective warping."""
    # Defined trapezoid based on standard dashcam perspective
    src = np.float32([[200, 720], [1100, 720], [595, 450], [685, 450]])
    dst = np.float32([[300, 720], [980, 720], [300, 0], [980, 0]])
    M = cv2.getPerspectiveTransform(src, dst)
    Minv = cv2.getPerspectiveTransform(dst, src)
    return M, Minv

3. Edge and Line Detection

This implements the Canny and Hough Transform steps to find lane candidates.

In [ ]:
def detect_lanes(frame):
    """Extracts Hough lines from a frame after Canny edge detection and ROI masking."""
    # 1. Preprocess (Color Masking + Blur)
    processed = preprocess_frame(frame)
    
    # 2. Canny Edge Detection 
    # Thresholds can be lower now because the noise was reduced by the color mask
    low_threshold = 40
    high_threshold = 120
    edges = cv2.Canny(processed, low_threshold, high_threshold)
    
    # 3. Mask ROI 
    masked_edges = get_roi_mask(edges)
    
    # 4. Hough Line Transform 
    rho = 1              
    theta = np.pi/180    
    threshold = 15       # Slightly lower threshold to catch fainter segments
    min_line_len = 10    
    max_line_gap = 250   
    
    lines = cv2.HoughLinesP(masked_edges, rho, theta, threshold, np.array([]),
                            minLineLength=min_line_len, maxLineGap=max_line_gap)
    
    return lines

def fit_polynomial_ransac(binary_warped):
    """Fits a 2nd-order polynomial (x = Ay^2 + By + C) to binary pixels using RANSAC."""
    # Extract pixel positions
    nonzero = binary_warped.nonzero()
    nonzeroy = np.array(nonzero[0])
    nonzerox = np.array(nonzero[1])
    
    if len(nonzerox) < 50: # Threshold for detection
        return None

    # Reshape for RANSAC (fitting x as a function of y)
    X = nonzeroy.reshape(-1, 1)
    y = nonzerox
    
    # Use RANSAC to fit a 2nd order polynomial (y = Ay^2 + By + C)
    # Note: We can expand X to [y^2, y] for a second-order fit
    X_poly = np.column_stack((nonzeroy**2, nonzeroy))
    
    ransac = RANSACRegressor()
    ransac.fit(X_poly, y)
    
    # Coefficients: [A, B] and Intercept: C
    A, B = ransac.estimator_.coef_
    C = ransac.estimator_.intercept_
    
    return np.array([A, B, C])

def fit_ransac(binary_warped):
    """Alternative RANSAC implementation for fitting polynomial coefficients to lane pixels."""
    # Find all non-zero pixels
    nonzero = binary_warped.nonzero()
    y_coords, x_coords = np.array(nonzero[0]), np.array(nonzero[1])
    
    if len(x_coords) < 100: return None
    
    # RANSAC fitting for 2nd order polynomial: x = Ay^2 + By + C
    X_poly = np.column_stack((y_coords**2, y_coords))
    ransac = RANSACRegressor()
    try:
        ransac.fit(X_poly, x_coords)
        return np.append(ransac.estimator_.coef_, ransac.estimator_.intercept_)
    except:
        return None

5. Temporal Filtering

The LaneTracker class uses a Kalman Filter to smooth the lane detection across consecutive frames, preventing the "flickering" effect caused by shadows or pavement changes.

In [ ]:
class LaneTracker:
    def __init__(self):
        self.kf = cv2.KalmanFilter(3, 3)
        self.kf.transitionMatrix = np.eye(3, dtype=np.float32)
        self.kf.measurementMatrix = np.eye(3, dtype=np.float32)
        self.kf.processNoiseCov = np.eye(3, dtype=np.float32) * 1e-4
        self.kf.measurementNoiseCov = np.eye(3, dtype=np.float32) * 1e-2
        self.kf.errorCovPost = np.eye(3, dtype=np.float32)

    def update(self, coeffs):
        if coeffs is not None:
            self.kf.correct(np.array(coeffs, dtype=np.float32))
        return self.kf.predict().flatten()

6. Main Pipeline Execution

The final loop integrates all components: color masking, warping, fitting, filtering, and drawing the green lane overlay back onto the original video.

In [68]:

print("Processing video... Please wait.") 
cap = cv2.VideoCapture(current_video)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
output_name = f"{video_file.split('.')[0]}_detection_v3.mp4"

M, Minv = get_warp_matrices((frame_height, frame_width))
left_track, right_track = LaneTracker(), LaneTracker()

# Video Writer Setup
fourcc = cv2.VideoWriter_fourcc(*'mp4v') 
out = cv2.VideoWriter(output_name, fourcc, fps, (frame_width, frame_height))
print("Processing stabilized pipeline... Please wait.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
        
    # --- 1. Robust Color Selection (HLS) ---
    hls = cv2.cvtColor(frame, cv2.COLOR_BGR2HLS)
    white_mask = cv2.inRange(hls, np.array([0, 200, 0]), np.array([180, 255, 255]))
    yellow_mask = cv2.inRange(hls, np.array([15, 0, 100]), np.array([35, 255, 255]))
    combined_binary = cv2.bitwise_or(white_mask, yellow_mask)

    # --- 2. Perspective Transform (Bird's Eye View) ---
    warped = cv2.warpPerspective(combined_binary, M, (frame_width, frame_height))

    # --- 3. Robust Fitting with RANSAC ---
    # Split image to find left/right lanes
    mid = frame_width // 2
    l_coeffs_raw = fit_ransac(warped[:, :mid])
    r_coeffs_raw = fit_ransac(warped[:, mid:])

    # --- 4. Temporal Stability via Kalman Filters ---
    l_fit = left_track.update(l_coeffs_raw)
    r_fit = right_track.update(r_coeffs_raw)

    # --- 5. Draw and Inverse Warp ---
    ploty = np.linspace(0, frame_height-1, frame_height)
    left_fitx = l_fit[0]*ploty**2 + l_fit[1]*ploty + l_fit[2]
    right_fitx = r_fit[0]*ploty**2 + r_fit[1]*ploty + (r_fit[2] + mid)

    # Create visualization overlay
    lane_overlay = np.zeros_like(frame)
    pts_left = np.array([np.transpose(np.vstack([left_fitx, ploty]))])
    pts_right = np.array([np.flipud(np.transpose(np.vstack([right_fitx, ploty])))])
    
    # FIXED: Use np.array with an explicit dtype instead of np.cast
    pts = np.array(np.hstack((pts_left, pts_right)), dtype=np.int32)
    
    cv2.fillPoly(lane_overlay, pts, (0, 255, 0)) # Green lane area
    newwarp = cv2.warpPerspective(lane_overlay, Minv, (frame_width, frame_height))
    final_frame = cv2.addWeighted(frame, 1, newwarp, 0.3, 0)
    
    out.write(final_frame)

cap.release()
out.release()
print(f"Success! Stabilized video saved as '{output_name}'")

Processing video... Please wait.
Processing stabilized pipeline... Please wait.
Success! Stabilized video saved as 'harder_challenge_video_detection_v3.mp4'
